## Лабораторная работа №1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark.
## Выполнил студент группы 6401-010302D, Стрельников Никита

## Задание

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

Импорты

In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.window import Window

In [2]:
spark = (
    SparkSession
        .builder
        .master("local[*]")
        .appName("LangsReport")
        .getOrCreate()
)
spark

#### Выборка 'posts_sample.xml'

In [3]:
posts = (
    spark
        .read
        .format('xml')
        .option('rowTag', 'row')
        .option("timestampFormat", 'y/M/d H:m:s')
        .load('posts_sample.xml')
)
posts.printSchema()

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: string (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: string (nullable = true)
 |-- _CreationDate: string (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: string (nullable = true)
 |-- _LastEditDate: string (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)



In [4]:
print("Описание датасета")
posts.describe().show()

Описание датасета
+-------+--------------------+------------------+--------------------+--------------------+-----------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+----------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+------------------+
|summary|   _AcceptedAnswerId|      _AnswerCount|               _Body|         _ClosedDate|    _CommentCount| _CommunityOwnedDate|       _CreationDate|    _FavoriteCount|                 _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName| _LastEditorUserId|_OwnerDisplayName|      _OwnerUserId|           _ParentId|       _PostTypeId|            _Score|            _Tags|              _Title|        _ViewCount|
+-------+--------------------+------------------+--------------------+--------------------+-----------------+-------------

In [5]:
print("Количество элементов: ", posts.count())

Количество элементов:  46006


#### Устанавливаем границы поиска

In [6]:
# Устанавливаем границы: 1 января 2010 года - 31 декабря 2020 года.
date_begin, date_end = "2010-01-01", "2020-12-31"
dates = (date_begin, date_end)

filtered_posts = (
    posts
        .filter(F.col("_CreationDate").between(*dates))
        .select(
            F.col("_CreationDate").alias("creation_date"),
            "*"
        )
)
filtered_posts.show(20)

+--------------------+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|       creation_date|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+--------------------+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+----------

#### Выборка 'programming-languages.csv'

In [7]:
languages = (
    spark
        .read
        .format('csv')
        .option('header', 'true')
        .option("inferSchema", True)
        .load('programming-languages.csv')
        .dropna()
)

In [8]:
languages.printSchema()

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)



In [9]:
print("Описание датасета")
languages.describe().show()

Описание датасета
+-------+--------+--------------------+
|summary|    name|       wikipedia_url|
+-------+--------+--------------------+
|  count|     699|                 699|
|   mean|    NULL|                NULL|
| stddev|    NULL|                NULL|
|    min|@Formula|https://en.wikipe...|
|    max|xHarbour|https://en.wikipe...|
+-------+--------+--------------------+



In [10]:
print("Количество элементов: ", languages.count())

Количество элементов:  699


#### Выводим названия языков программирования

In [11]:
language_names = [str(x[0]) for x in languages.collect()]
language_names

['A# .NET',
 'A# (Axiom)',
 'A-0 System',
 'A+',
 'A++',
 'ABAP',
 'ABC',
 'ABC ALGOL',
 'ABSET',
 'ABSYS',
 'ACC',
 'Accent',
 'Ace DASL',
 'ACL2',
 'ACT-III',
 'Action!',
 'ActionScript',
 'Ada',
 'Adenine',
 'Agda',
 'Agilent VEE',
 'Agora',
 'AIMMS',
 'Alef',
 'ALF',
 'ALGOL 58',
 'ALGOL 60',
 'ALGOL 68',
 'ALGOL W',
 'Alice',
 'Alma-0',
 'AmbientTalk',
 'Amiga E',
 'AMOS',
 'AMPL',
 'Apex (Salesforce.com)',
 'APL',
 "App Inventor for Android's visual block language",
 'AppleScript',
 'Arc',
 'ARexx',
 'Argus',
 'AspectJ',
 'Assembly language',
 'ATS',
 'Ateji PX',
 'AutoHotkey',
 'Autocoder',
 'AutoIt',
 'AutoLISP / Visual LISP',
 'Averest',
 'AWK',
 'Axum',
 'B',
 'Babbage',
 'Bash',
 'BASIC',
 'bc',
 'BCPL',
 'BeanShell',
 'Batch (Windows/Dos)',
 'Bertrand',
 'BETA',
 'Bigwig',
 'Bistro',
 'BitC',
 'BLISS',
 'Blockly',
 'BlooP',
 'Blue',
 'Boo',
 'Boomerang',
 'Bourne shell (including',
 'bash and',
 'ksh )',
 'BREW',
 'BPEL',
 'C',
 'C--',
 'C++ – ISO/IEC 14882',
 'C# – ISO/IEC

In [12]:
filtered_languages = languages.select(
    F.lower(F.trim(F.col("name"))).alias("language"),
    "wikipedia_url"
).dropDuplicates(["language"])
filtered_languages.show(10)

+----------+--------------------+
|  language|       wikipedia_url|
+----------+--------------------+
|  @formula|https://en.wikipe...|
|a# (axiom)|https://en.wikipe...|
|   a# .net|https://en.wikipe...|
|        a+|https://en.wikipe...|
|       a++|https://en.wikipe...|
|a-0 system|https://en.wikipe...|
|      abap|https://en.wikipe...|
|       abc|https://en.wikipe...|
| abc algol|https://en.wikipe...|
|     abset|https://en.wikipe...|
+----------+--------------------+
only showing top 10 rows


#### Преобразование строки тегов вида `<java><spring><hibernate>` в отдельные теги

In [13]:
posts_tags = (
    filtered_posts
        .select(
            F.col("_Tags").alias("tags"),
            "*"
        )
        .withColumn(
            "tag",
            F.explode(
                F.split(
                    F.regexp_replace(F.col("tags"), r"[<>]", " "),
                    r"\s+"
                )
            )
        )
        .filter(F.col("tag") != "")
)

#### Оставляем только те теги, которые являются языками программирования

In [14]:
language_mentions = posts_tags.join(
    F.broadcast(filtered_languages),
    posts_tags.tag == filtered_languages.language,
    "inner"
).select(
    posts_tags.creation_date,
    filtered_languages.language
)

#### Подсчёт количества упоминаний языков по годам

In [15]:
year_language_counts = (
    language_mentions
        .groupBy("creation_date", "language")
        .agg(F.count("*").alias("mentions")
    )
)

#### Формирование топ-10 языков для каждого года

In [16]:
window_spec = Window.partitionBy("creation_date").orderBy(F.desc("mentions"), F.asc("language"))

top10_per_year = year_language_counts.withColumn(
    "rank",
    F.row_number().over(window_spec)
).filter(
    F.col("rank") <= 10
).orderBy("creation_date", "rank")

In [17]:
top10_per_year.show(200, truncate=False)

+-----------------------+------------+--------+----+
|creation_date          |language    |mentions|rank|
+-----------------------+------------+--------+----+
|2010-01-05T13:04:18.630|ruby        |1       |1   |
|2010-01-06T13:09:09.707|python      |1       |1   |
|2010-01-08T12:00:14.930|javascript  |1       |1   |
|2010-01-10T22:30:52.080|objective-c |1       |1   |
|2010-01-12T10:41:18.467|javascript  |1       |1   |
|2010-01-13T02:32:31.053|ruby        |1       |1   |
|2010-01-14T00:16:27.470|c           |1       |1   |
|2010-01-16T04:58:40.517|mouse       |1       |1   |
|2010-01-17T09:14:30.063|javascript  |1       |1   |
|2010-01-18T12:42:49.313|php         |1       |1   |
|2010-01-18T20:38:35.333|python      |1       |1   |
|2010-01-19T20:28:54.203|javascript  |1       |1   |
|2010-01-19T21:32:29.063|applescript |1       |1   |
|2010-01-21T08:46:02.870|python      |1       |1   |
|2010-01-21T16:56:19.483|ruby        |1       |1   |
|2010-01-22T20:56:31.550|objective-c |1       

#### Сохранение отчёта в Apache Parquet

In [18]:
top10_per_year.write.mode("overwrite").parquet("top10_languages_2010_2020.parquet")

In [19]:
spark.stop()